In [1]:
from dotenv import load_dotenv 
import os
load_dotenv()


True

In [8]:
from langchain_groq import ChatGroq     
api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=api_key)
llm.invoke("Hello World!")

AIMessage(content="Hello World! It's nice to see you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 38, 'total_tokens': 64, 'completion_time': 0.033212381, 'completion_tokens_details': None, 'prompt_time': 0.002766411, 'prompt_tokens_details': None, 'queue_time': 0.088341987, 'total_time': 0.035978792}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b892b-1eb7-7572-9f55-ff17b43ea837-0', usage_metadata={'input_tokens': 38, 'output_tokens': 26, 'total_tokens': 64})

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [9]:
speech="""Good morning everyone.

Today, I want to talk about how small changes can create a big impact. Many of us wait for the perfect time to start something new, but that moment rarely comes. Instead, progress begins when we take simple, consistent steps.

In our team, we started by improving just one process at a time. These small improvements reduced errors, saved time, and increased confidence. Over time, the results added up and transformed the way we work.

This experience taught us that success is not about dramatic change, but about steady effort. If each of us commits to small improvements every day, we can achieve meaningful and lasting growth.

Thank you. """

In [13]:
chat_message =[SystemMessage(content="You are a expert in summarizing speeches."),
               HumanMessage(content=f"Summarize the following speech in 2-3 lines:\n Text:{speech}")]

In [15]:
llm.invoke(chat_message).content

"Here is a 2-3 line summary of the speech:\n\nThe speaker emphasizes that small, consistent changes can create a significant impact. By improving one process at a time, they achieved lasting growth and transformed their team's work. Success comes from steady effort, not dramatic change."

### Prompt Template for small content

In [21]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a expert in summarizing speeches.
            Summarize the following speech in 2-3 lines:
            Text: {speech} 
            Translate the summary into {language} language."""


prompt = ChatPromptTemplate.from_template(template)


In [22]:
chain = prompt | llm
result = chain.invoke({
    "speech": speech, 
    "language": 'Spanish'
})
print(result.content)

Here is a 2-3 line summary of the speech:

The speaker emphasizes that small, consistent changes can have a significant impact. By improving one process at a time, their team achieved noticeable results and transformed the way they work. Success is about steady effort, not dramatic change.

And here is the summary translated into Spanish:

El hablante destaca que los pequeños cambios consistentes pueden tener un impacto significativo. Al mejorar un proceso a la vez, su equipo logró resultados notables y transformó la forma en que trabajan. El éxito se trata de un esfuerzo constante, no de un cambio dramático.


### Stuff Document Chain for Summarization 

In [27]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("apjspeech.pdf")
doc=loader.load_and_split() 
doc[0].page_content
len(doc)

7

In [28]:
from langchain_core.prompts import ChatPromptTemplate
template=""" Write a concise summary of the following speech: {text}

"""
prompt=ChatPromptTemplate.from_template(template)


In [30]:
from langchain_classic.chains.summarize import load_summarize_chain

chain=load_summarize_chain(llm,chain_type="stuff",prompt=prompt,verbose=True)
chain.invoke({"input_documents":doc}) 




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human:  Write a concise summary of the following speech: A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions,

{'input_documents': [Document(metadata={'producer': 'GPL Ghostscript 8.15', 'creator': 'PScript5.dll Version 5.2', 'creationdate': 'D:20070730160943', 'moddate': 'D:20070730160943', 'title': 'Microsoft Word - Document1', 'author': 'Shri', 'source': 'apjspeech.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who 

### Map Reduce Summarization 

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
final_document=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100).split_documents(doc)

In [35]:
len(final_document)

22

In [38]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.summarize import load_summarize_chain

template=""" Write a concise summary of the following speech: {text} 
Summary: 
""" 
chunks_prompt=ChatPromptTemplate.from_template(template)

final_prompt=""" Provide the final summary of the entire speech with these import points.
Add a motivation Title,Start the precise summary with an introduction and provide the summary in number points for the speech.
Speech Points: {text} """ 
 
final_prompt= ChatPromptTemplate.from_template(final_prompt) # prompt to combine all the chunks summary. 

chain=load_summarize_chain(llm,chain_type="map_reduce",map_prompt=chunks_prompt,combine_prompt=final_prompt ,verbose=True) 
output=chain.invoke({"input_documents":final_document}) 



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human:  Write a concise summary of the following speech: A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interacti

In [39]:
chain=load_summarize_chain(llm,chain_type="refine",verbose=True) 

output=chain.invoke({"input_documents":final_document}) 






> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have man

In [43]:
!pip install numexpr